In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def get_rmat(yaw):
    y = np.array([0, 0, -1])
    z = np.array([-np.cos(yaw), -np.sin(yaw), 0])
    return np.array([np.cross(y, z), y, z])


f = 22388.125
f_mm = 107.463
w = 1024
h = 512
cx = (w - 1) / 2
cy = (h - 1) / 2
rmats_gt = np.array(
    [get_rmat(theta) for theta in np.deg2rad([-120, -90, -45, 0, 45, 90, 120])]
)
tvecs_gt = np.array([[0, 0, f_mm]] * len(rmats_gt), dtype=float)
pts3d_gt = np.random.default_rng(0).uniform(-0.5, 0.5, size=(10, 3))

In [ ]:
def plot_camera(rmat, tvec, ax3d, length=None):
    center = -rmat.T @ tvec
    if length is None:
        length = np.linalg.norm(tvec) * 0.2
    for i, c in enumerate("rgb"):
        ax3d.plot(*zip(center, center + rmat[i] * length), c=c)


fig, ax3d = plt.subplots(subplot_kw={"projection": "3d"})
for rmat, tvec in zip(rmats_gt, tvecs_gt):
    plot_camera(rmat, tvec, ax3d)
ax3d.set_aspect("equal")

In [ ]:
import numpy as np
from scipy.linalg import expm


def small_rotation(sigma=1e-3, seed=None):
    rng = np.random.default_rng(seed)
    omega = rng.normal(scale=sigma, size=3)  # small rotation vector
    K = np.array(
        [[0, -omega[2], omega[1]], [omega[2], 0, -omega[0]], [-omega[1], omega[0], 0]]
    )
    return expm(K)


rmats = np.array([small_rotation(0.05, i) @ rmat for i, rmat in enumerate(rmats_gt)])
tvecs = tvecs_gt + np.random.default_rng(0).normal(scale=5, size=tvecs_gt.shape)

fig, ax3d = plt.subplots(subplot_kw={"projection": "3d"})
for rmat, tvec in zip(rmats_gt, tvecs_gt):
    plot_camera(rmat, tvec, ax3d)
for rmat, tvec in zip(rmats, tvecs):
    plot_camera(rmat, tvec, ax3d)
ax3d.set_aspect("equal")

In [ ]:
from deeperfly.multiview_geom import rmats2rvecs

rvecs_gt = rmats2rvecs(rmats_gt)